# Phases, Lattices, Space Groups, And CIF Import

PyTex keeps structure semantics explicit by separating point-group reduction
semantics from structure-facing space-group identity. The immediate teaching path
now starts from the pinned in-repo phase-fixture corpus so the examples line up
with the structure-import validation and reproducibility surface.

## Key Rule

- `SymmetrySpec` is the orientation-reduction surface
- `SpaceGroupSpec` is the structure-definition surface
- `Phase` owns both when the structure source provides them


In [1]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    normalize_ebsd,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_ipf_map,
    plot_orientations,
    plot_kam_map,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [2]:
crystal, specimen, map_frame, detector, lab, phase = make_context()
ni_fixture = describe_phase_fixture("ni_fcc")

print("Fixture ids:", [record.fixture_id for record in list_phase_fixtures()])
print("Teaching fixture:", ni_fixture["display_name"])
print("Phase:", phase.name)
print("Point group:", phase.symmetry.point_group)
print("Direct basis:")
print(phase.lattice.direct_basis().matrix)
print("Reciprocal basis:")
print(phase.lattice.reciprocal_basis().matrix)


Fixture ids: ['fe_bcc', 'zr_hcp', 'ni_fcc', 'nicl', 'diamond']
Teaching fixture: Nickel (FCC)
Phase: nickel-fcc
Point group: m-3m
Direct basis:
[[3.52387000e+00 2.15774806e-16 2.15774806e-16]
 [0.00000000e+00 3.52387000e+00 2.15774806e-16]
 [0.00000000e+00 0.00000000e+00 3.52387000e+00]]
Reciprocal basis:
[[ 2.83778914e-01  0.00000000e+00  0.00000000e+00]
 [-1.73764469e-17  2.83778914e-01  0.00000000e+00]
 [-1.73764469e-17 -1.73764469e-17  2.83778914e-01]]


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:720: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn(msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:1224: UserWarning: Issues encountered while parsing CIF: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


In [3]:
zr_hcp = load_zr_hcp_phase()
structure_audit = read_workflow_result_manifest(
    "benchmarks/structure_import/foundation_workflow_result_manifest.json"
).to_dict()

print("Hexagonal fixture:", zr_hcp.name)
print(zr_hcp.space_group.symbol, zr_hcp.space_group.number)
print("Audit summary artifact:", structure_audit["metadata"]["audit_summary_artifact"])


Hexagonal fixture: zirconium-hcp
P6_3/mmc 194
Audit summary artifact: benchmarks/structure_import/phase_fixture_audit_summary.json


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:720: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn(msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:1224: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


In [4]:
nacl_cif = """
data_NaCl
_symmetry_space_group_name_H-M 'F m -3 m'
_symmetry_Int_Tables_number 225
_cell_length_a 5.6402
_cell_length_b 5.6402
_cell_length_c 5.6402
_cell_angle_alpha 90
_cell_angle_beta 90
_cell_angle_gamma 90
loop_
  _atom_site_label
  _atom_site_type_symbol
  _atom_site_fract_x
  _atom_site_fract_y
  _atom_site_fract_z
  Na1 Na 0.0 0.0 0.0
  Cl1 Cl 0.5 0.5 0.5
"""

try:
    imported_phase = Phase.from_cif_string(nacl_cif, crystal_frame=make_context()[0])
    print(imported_phase.name)
    print(imported_phase.space_group.symbol, imported_phase.space_group.number)
    print(imported_phase.symmetry.point_group)
except ImportError:
    print("Install the default development environment to execute CIF-backed examples.")


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:720: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn(msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:1224: UserWarning: Issues encountered while parsing CIF: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:720: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn(msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:1224: UserWarning: Issues encountered while parsing CIF: No _symmetry_equiv_pos_as_xyz 

NaCl
Fm-3m 225
m-3m


The fixture-backed examples above are the default reproducible path. The inline CIF
string remains useful for tiny teaching snippets, but the pinned fixture corpus and
audit manifest are what the repository treats as the authoritative validation
surface.
